# Notebook 09 — Data Preparation (Threshold = 50 Reviews)
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Purpose:** Re-run the NB02 data preparation pipeline with a lower review threshold
(50 instead of 100) to see if a larger and more balanced dataset is produced.

**What changes from NB02:** Only `MIN_REVIEWS = 50` (overrides config). Every
parsing function, cap, feature, and encoding is otherwise identical so the output
has exactly the same 53 feature columns as `games_features_t4.csv`.

**What does NOT change:** Raw data path, column names, caps, derived features,
genre/tag/category lists, label threshold — all identical to NB02.

**Output:** `data/processed/games_features_t4_threshold50.csv`

The original `games_features_t4.csv` is not modified.

---
## Contents
1. Setup and Imports
2. Load Raw Dataset
3. Filter and Create Label
4. Drop Unwanted Columns
5. Fix Broken Values and Cap Outliers
6. Create Derived Features
7. Parse Count Features
8. Parse and Encode Genres
9. Parse and Encode Tags
10. Parse and Encode Categories
11. Final Cleanup
12. Verification Checks and Save
13. Comparison vs Original (100-review threshold)

## 1. Setup and Imports

In [1]:
import sys
import ast
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Navigate from notebooks/ up to the project root — same as NB02 ────────────
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# ── Import everything from config.py exactly as NB02 does ────────────────────
from src.config import (
    RAW_CSV, FEATURES_T4_CSV, FILTERED_CSV,
    MIN_REVIEW_COUNT, POSITIVE_THRESHOLD, TARGET_COL,
    COL_LABEL, COL_FILTER, COL_PRICE, COL_AGE, COL_DLC,
    COL_GENRES, COL_TAGS, COL_CATS, COL_MAC, COL_LINUX,
    COL_LANGS, COL_SCREENSHOTS, COL_MOVIES,
    COL_ACHIEVEMENTS, COL_WEBSITE, COL_DESC,
    DLC_CAP, AGE_SENTINEL, SCREENSHOT_CAP, MOVIE_CAP, LANG_COUNT_CAP,
    TOP_10_GENRES, TOP_20_TAGS, KEY_CATEGORIES,
    RANDOM_STATE
)

# ── The only override — lower review threshold ────────────────────────────────
MIN_REVIEWS = 50   # NB02 uses MIN_REVIEW_COUNT = 100 from config; we override here

# ── Output path — different filename so original is never touched ─────────────
OUTPUT_CSV = PROJECT_ROOT / 'data' / 'processed' / 'games_features_t4_threshold50.csv'
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print('Imports OK')
print(f'Project root   : {PROJECT_ROOT}')
print(f'Raw CSV        : {RAW_CSV}')
print(f'Output CSV     : {OUTPUT_CSV}')
print(f'Min reviews    : {MIN_REVIEWS}  (NB02 used {MIN_REVIEW_COUNT})')
print(f'Label threshold: {POSITIVE_THRESHOLD}  (unchanged)')

Imports OK
Project root   : C:\Users\3214h\Documents\fyp-steam-reception
Raw CSV        : C:\Users\3214h\Documents\fyp-steam-reception\data\raw\Dataset-2.csv
Output CSV     : C:\Users\3214h\Documents\fyp-steam-reception\data\processed\games_features_t4_threshold50.csv
Min reviews    : 50  (NB02 used 100)
Label threshold: 0.75  (unchanged)


## 2. Load Raw Dataset

In [2]:
assert RAW_CSV.exists(), f'Dataset not found at {RAW_CSV}. Download from Kaggle.'
df_raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Raw dataset loaded: {len(df_raw):,} rows x {df_raw.shape[1]} columns')

Raw dataset loaded: 89,618 rows x 47 columns


## 3. Filter and Create Label

Same three steps as NB02 — only the threshold number changes.

In [3]:
# ── Step 1: Remove -1 sentinel rows ───────────────────────────────────────────
df_raw[COL_FILTER] = pd.to_numeric(df_raw[COL_FILTER], errors='coerce')
df = df_raw[df_raw[COL_FILTER] > 0].copy()
print(f'After -1 sentinel removal  : {len(df):,} games')

# ── Step 2: Apply >= 50 review filter (changed from 100) ─────────────────────
df = df[df[COL_FILTER] >= MIN_REVIEWS].copy()
print(f'After >= {MIN_REVIEWS} review filter    : {len(df):,} games  ({len(df)/len(df_raw)*100:.1f}% of raw)')
print(f'(NB02 with >= 100 reviews  : 20,383 games  (22.7% of raw))')
print(f'Additional games retained  : {len(df) - 20383:,}')

# ── Step 3: Create binary label ────────────────────────────────────────────────
df[COL_LABEL] = pd.to_numeric(df[COL_LABEL], errors='coerce')
df['positive_ratio'] = df[COL_LABEL] / 100.0
df[TARGET_COL] = (df['positive_ratio'] >= POSITIVE_THRESHOLD).astype(int)

counts = df[TARGET_COL].value_counts()
print(f'\nClass distribution:')
print(f'  Well Received     (1): {counts.get(1,0):>6,}  ({counts.get(1,0)/len(df)*100:.1f}%)')
print(f'  Not Well Received (0): {counts.get(0,0):>6,}  ({counts.get(0,0)/len(df)*100:.1f}%)')
print(f'\nNB02 baseline: 14,631 (71.8%) / 5,752 (28.2%)')
print(f'Additional Not Well Received games: {counts.get(0,0) - 5752:,}')

After -1 sentinel removal  : 53,199 games
After >= 50 review filter    : 27,750 games  (31.0% of raw)
(NB02 with >= 100 reviews  : 20,383 games  (22.7% of raw))
Additional games retained  : 7,367

Class distribution:
  Well Received     (1): 19,002  (68.5%)
  Not Well Received (0):  8,748  (31.5%)

NB02 baseline: 14,631 (71.8%) / 5,752 (28.2%)
Additional Not Well Received games: 2,996


## 4. Drop Unwanted Columns

In [4]:
# ── Same keep-list as NB02 ───────────────────────────────────────────────────
KEEP_COLS = [
    TARGET_COL,
    'positive_ratio',
    COL_PRICE,
    COL_AGE,
    COL_DLC,
    COL_GENRES,
    COL_TAGS,
    COL_CATS,
    COL_MAC,
    COL_LINUX,
    COL_LANGS,
    COL_SCREENSHOTS,
    COL_MOVIES,
    COL_ACHIEVEMENTS,
    COL_WEBSITE,
    COL_DESC,
]

missing = [c for c in KEEP_COLS if c not in df.columns]
if missing:
    print(f'WARNING — columns not found: {missing}')
else:
    print('All expected columns present ✓')

df = df[KEEP_COLS].copy()
print(f'Shape after column drop: {df.shape[1]} columns remaining')

All expected columns present ✓
Shape after column drop: 16 columns remaining


## 5. Fix Broken Values and Cap Outliers

In [5]:
# ── Ensure numerical columns are correct dtype — same as NB02 ────────────────
for col in [COL_PRICE, COL_AGE, COL_DLC, COL_ACHIEVEMENTS]:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# ── Fix required_age sentinel (-1 means age not set — replace with 0) ─────────
n_sentinel = (df[COL_AGE] == AGE_SENTINEL).sum()
df[COL_AGE] = df[COL_AGE].replace(AGE_SENTINEL, 0)
print(f'required_age: replaced {n_sentinel} sentinel (-1) values with 0')
print(f'  New range: {df[COL_AGE].min()} to {df[COL_AGE].max()}')

# ── Cap dlc_count at 50 — same cap as NB02 ────────────────────────────────────
n_above_dlc = (df[COL_DLC] > DLC_CAP).sum()
df[COL_DLC] = df[COL_DLC].clip(0, DLC_CAP)
print(f'\ndlc_count: capped {n_above_dlc} games above {DLC_CAP}')

print(f'\nprice: min={df[COL_PRICE].min()}, max={df[COL_PRICE].max():.2f}, median={df[COL_PRICE].median():.2f}')
print('Outlier fixes applied ✓')

required_age: replaced 1 sentinel (-1) values with 0
  New range: 0 to 18

dlc_count: capped 63 games above 50

price: min=0.0, max=89.99, median=5.99
Outlier fixes applied ✓


## 6. Create Derived Features

In [6]:
# ── Same 6 derived features as NB02 ──────────────────────────────────────────

# is_free
df['is_free'] = (df[COL_PRICE] == 0.0).astype(int)
print(f'is_free: {df["is_free"].sum():,} games ({df["is_free"].mean()*100:.1f}%)')

# price_to_dlc_ratio
df['price_to_dlc_ratio'] = df[COL_PRICE] / (df[COL_DLC] + 1)
print(f'price_to_dlc_ratio: mean={df["price_to_dlc_ratio"].mean():.2f}, median={df["price_to_dlc_ratio"].median():.2f}')

# platform_coverage = (mac + linux) / 2 — windows excluded (zero variance)
df[COL_MAC]   = df[COL_MAC].map(lambda x: 1 if str(x).strip().lower() in ('true','1','yes') else 0)
df[COL_LINUX] = df[COL_LINUX].map(lambda x: 1 if str(x).strip().lower() in ('true','1','yes') else 0)
df['platform_coverage'] = (df[COL_MAC] + df[COL_LINUX]) / 2
print(f'platform_coverage: {df["platform_coverage"].value_counts().sort_index().to_dict()}')

# has_achievements (binary — raw count is insignificant, binary is significant)
df['has_achievements'] = (df[COL_ACHIEVEMENTS] > 0).astype(int)
print(f'has_achievements: {df["has_achievements"].sum():,} games ({df["has_achievements"].mean()*100:.1f}%)')

# has_website
df['has_website'] = df[COL_WEBSITE].apply(
    lambda x: 0 if (pd.isna(x) or str(x).strip() == '') else 1
)
print(f'has_website: {df["has_website"].sum():,} games ({df["has_website"].mean()*100:.1f}%)')

# genre_concentration — placeholder, set properly in Section 8 after genre parsing
df['genre_concentration'] = 0.0
print('\nAll derived features created ✓')

is_free: 5,962 games (21.5%)
price_to_dlc_ratio: mean=6.74, median=4.99
platform_coverage: {0.0: 19089, 0.5: 4239, 1.0: 4422}
has_achievements: 19,649 games (70.8%)
has_website: 16,800 games (60.5%)

All derived features created ✓


## 7. Parse Count Features

In [7]:
# ── Same parsing functions as NB02 ───────────────────────────────────────────

def safe_list_len(val):
    """Parse a string-encoded list/dict and return its length. Returns 0 for null or unparseable."""
    if pd.isna(val) or str(val).strip() in ('', '[]', '{}'):
        return 0
    try:
        result = ast.literal_eval(str(val))
        if isinstance(result, (list, dict)):
            return len(result)
        return 0
    except Exception:
        return 0


def count_languages(val):
    """Parse supported_languages column. Handles list format and comma-separated strings."""
    if pd.isna(val) or str(val).strip() in ('', '[]'):
        return 0
    try:
        result = ast.literal_eval(str(val))
        return len(result) if isinstance(result, list) else 1
    except Exception:
        return len([x for x in str(val).split(',') if x.strip()])


# ── supported_languages_count (cap at 30) ─────────────────────────────────────
df['supported_languages_count'] = df[COL_LANGS].apply(count_languages)
n_before = (df['supported_languages_count'] > LANG_COUNT_CAP).sum()
df['supported_languages_count'] = df['supported_languages_count'].clip(0, LANG_COUNT_CAP)
print(f'supported_languages_count: median={df["supported_languages_count"].median():.0f}, '
      f'max={df["supported_languages_count"].max()}, capped {n_before} games above {LANG_COUNT_CAP}')

# ── screenshot_count (cap at 20 — Steam hard limit) ───────────────────────────
df['screenshot_count'] = df[COL_SCREENSHOTS].apply(safe_list_len)
n_before = (df['screenshot_count'] > SCREENSHOT_CAP).sum()
df['screenshot_count'] = df['screenshot_count'].clip(0, SCREENSHOT_CAP)
print(f'screenshot_count: median={df["screenshot_count"].median():.0f}, '
      f'max={df["screenshot_count"].max()}, capped {n_before} games above {SCREENSHOT_CAP}')

# ── movie_count (cap at 10) ────────────────────────────────────────────────────
df['movie_count'] = df[COL_MOVIES].apply(safe_list_len)
n_before = (df['movie_count'] > MOVIE_CAP).sum()
df['movie_count'] = df['movie_count'].clip(0, MOVIE_CAP)
print(f'movie_count: median={df["movie_count"].median():.0f}, '
      f'max={df["movie_count"].max()}, capped {n_before} games above {MOVIE_CAP}')

print('\nAll count features parsed ✓')

supported_languages_count: median=3, max=30, capped 384 games above 30
screenshot_count: median=9, max=20, capped 1498 games above 20
movie_count: median=1, max=10, capped 81 games above 10

All count features parsed ✓


## 8. Parse and Encode Genres

In [8]:
def parse_list_col(series_val):
    """Parse a string-encoded list into a Python list. Used for genres and categories."""
    if pd.isna(series_val) or str(series_val).strip() in ('', '[]', '{}'):
        return []
    try:
        result = ast.literal_eval(str(series_val))
        if isinstance(result, list):
            return [str(x).strip() for x in result]
        elif isinstance(result, str):
            return [result.strip()]
        return []
    except Exception:
        items = [x.strip().strip("'\")") for x in str(series_val).split(',') if x.strip()]
        return items


# ── Parse genres ──────────────────────────────────────────────────────────────
df['_genres_parsed'] = df[COL_GENRES].apply(parse_list_col)

# ── genre_concentration — same formula as NB02: genres assigned / 10, clipped at 1.0
df['genre_concentration'] = (df['_genres_parsed'].apply(len) / 10).clip(0, 1.0)
print(f'genre_concentration: mean={df["genre_concentration"].mean():.3f}, median={df["genre_concentration"].median():.3f}')

# ── One-hot encode top 10 genres — same TOP_10_GENRES list as NB02 ────────────
print(f'\nEncoding {len(TOP_10_GENRES)} genres:')
for genre in TOP_10_GENRES:
    col_name = f'genre_{genre.replace(" ", "_").replace("-", "_")}'
    df[col_name] = df['_genres_parsed'].apply(lambda lst: 1 if genre in lst else 0)
    rate = df.loc[df[col_name] == 1, TARGET_COL].mean()
    print(f'  {col_name:<40} n={df[col_name].sum():>6,}  well_received={rate:.3f}')

print('Genre encoding complete ✓')

genre_concentration: mean=0.294, median=0.300

Encoding 10 genres:
  genre_Indie                              n=19,016  well_received=0.707
  genre_Adventure                          n=11,965  well_received=0.696
  genre_Action                             n=11,240  well_received=0.654
  genre_Casual                             n= 9,902  well_received=0.717
  genre_Simulation                         n= 7,106  well_received=0.616
  genre_RPG                                n= 6,472  well_received=0.638
  genre_Strategy                           n= 5,962  well_received=0.618
  genre_Free_To_Play                       n= 4,252  well_received=0.624
  genre_Early_Access                       n= 2,295  well_received=0.582
  genre_Massively_Multiplayer              n= 1,112  well_received=0.337
Genre encoding complete ✓


## 9. Parse and Encode Tags

In [9]:
def parse_tags_col(val):
    """Parse tags column (stored as dict with vote counts) into list of tag names."""
    if pd.isna(val) or str(val).strip() in ('', '[]', '{}'):
        return []
    try:
        result = ast.literal_eval(str(val))
        if isinstance(result, dict):
            return list(result.keys())
        elif isinstance(result, list):
            return [str(x).strip() for x in result]
        return []
    except Exception:
        return []


# ── Parse tags ────────────────────────────────────────────────────────────────
df['_tags_parsed'] = df[COL_TAGS].apply(parse_tags_col)
n_no_tags = (df['_tags_parsed'].apply(len) == 0).sum()
print(f'Games with no tags: {n_no_tags:,} ({n_no_tags/len(df)*100:.1f}%)')

# ── One-hot encode top 20 tags — same TOP_20_TAGS list as NB02 ────────────────
print(f'\nEncoding {len(TOP_20_TAGS)} tags:')
for tag in TOP_20_TAGS:
    col_name = f'tag_{tag.replace(" ", "_").replace("-", "_")}'
    df[col_name] = df['_tags_parsed'].apply(lambda lst: 1 if tag in lst else 0)
    rate = df.loc[df[col_name] == 1, TARGET_COL].mean()
    print(f'  {col_name:<40} n={df[col_name].sum():>6,}  well_received={rate:.3f}')

print('Tag encoding complete ✓')

Games with no tags: 4,049 (14.6%)

Encoding 20 tags:
  tag_Singleplayer                         n=15,030  well_received=0.727
  tag_Indie                                n=13,914  well_received=0.682
  tag_Adventure                            n=11,199  well_received=0.696
  tag_Action                               n=10,648  well_received=0.655
  tag_Casual                               n= 8,849  well_received=0.721
  tag_2D                                   n= 7,058  well_received=0.803
  tag_Simulation                           n= 6,366  well_received=0.616
  tag_RPG                                  n= 5,894  well_received=0.646
  tag_Atmospheric                          n= 5,818  well_received=0.730
  tag_Story_Rich                           n= 5,524  well_received=0.778
  tag_Strategy                             n= 5,827  well_received=0.619
  tag_Multiplayer                          n= 4,981  well_received=0.611
  tag_Puzzle                               n= 4,406  well_received=0.79

## 10. Parse and Encode Categories

In [10]:
# ── Parse categories ──────────────────────────────────────────────────────────
df['_cats_parsed'] = df[COL_CATS].apply(parse_list_col)

# ── One-hot encode key categories — same KEY_CATEGORIES list as NB02 ──────────
print(f'Encoding {len(KEY_CATEGORIES)} categories:')
for cat in KEY_CATEGORIES:
    col_name = f'cat_{cat.replace(" ", "_").replace("-", "_").replace("/", "_")}'
    df[col_name] = df['_cats_parsed'].apply(lambda lst: 1 if cat in lst else 0)
    rate = df.loc[df[col_name] == 1, TARGET_COL].mean()
    print(f'  {col_name:<45} n={df[col_name].sum():>6,}  well_received={rate:.3f}')

print('Category encoding complete ✓')

Encoding 12 categories:
  cat_Single_player                             n=26,242  well_received=0.700
  cat_Multi_player                              n= 7,058  well_received=0.592
  cat_Co_op                                     n= 4,023  well_received=0.609
  cat_Steam_Achievements                        n=17,967  well_received=0.720
  cat_Steam_Cloud                               n=11,572  well_received=0.767
  cat_Full_controller_support                   n= 8,160  well_received=0.761
  cat_Partial_Controller_Support                n= 4,099  well_received=0.677
  cat_Online_Co_op                              n= 2,751  well_received=0.569
  cat_Online_PvP                                n= 3,442  well_received=0.538
  cat_In_App_Purchases                          n= 1,600  well_received=0.405
  cat_Steam_Workshop                            n= 1,368  well_received=0.776
  cat_Remote_Play_Together                      n= 2,410  well_received=0.749
Category encoding complete ✓


## 11. Final Cleanup

In [11]:
# ── Drop raw source columns that have been encoded or consumed ───────────────
COLS_TO_DROP = [
    COL_GENRES, COL_TAGS, COL_CATS,
    COL_MAC, COL_LINUX,
    COL_LANGS, COL_SCREENSHOTS, COL_MOVIES,
    COL_ACHIEVEMENTS, COL_WEBSITE,
    '_genres_parsed', '_tags_parsed', '_cats_parsed',
    'positive_ratio',
]
cols_present = [c for c in COLS_TO_DROP if c in df.columns]
df.drop(columns=cols_present, inplace=True)
df = df.reset_index(drop=True)

print(f'Dropped {len(cols_present)} raw source columns')
print(f'Remaining columns ({df.shape[1]} total):')
for col in df.columns:
    print(f'  {col}')

Dropped 14 raw source columns
Remaining columns (56 total):
  reception_label
  price
  required_age
  dlc_count
  short_description
  is_free
  price_to_dlc_ratio
  platform_coverage
  has_achievements
  has_website
  genre_concentration
  supported_languages_count
  screenshot_count
  movie_count
  genre_Indie
  genre_Adventure
  genre_Action
  genre_Casual
  genre_Simulation
  genre_RPG
  genre_Strategy
  genre_Free_To_Play
  genre_Early_Access
  genre_Massively_Multiplayer
  tag_Singleplayer
  tag_Indie
  tag_Adventure
  tag_Action
  tag_Casual
  tag_2D
  tag_Simulation
  tag_RPG
  tag_Atmospheric
  tag_Story_Rich
  tag_Strategy
  tag_Multiplayer
  tag_Puzzle
  tag_Exploration
  tag_First_Person
  tag_Anime
  tag_Funny
  tag_3D
  tag_Cute
  tag_Fantasy
  cat_Single_player
  cat_Multi_player
  cat_Co_op
  cat_Steam_Achievements
  cat_Steam_Cloud
  cat_Full_controller_support
  cat_Partial_Controller_Support
  cat_Online_Co_op
  cat_Online_PvP
  cat_In_App_Purchases
  cat_Steam_Works

## 12. Verification Checks and Save

In [12]:
# ── Build expected column list — identical structure to NB02 ─────────────────
EXPECTED_COLS = (
    [TARGET_COL]
    + [COL_PRICE, COL_AGE, COL_DLC]
    + ['is_free', 'price_to_dlc_ratio', 'platform_coverage',
       'has_achievements', 'has_website', 'genre_concentration']
    + ['supported_languages_count', 'screenshot_count', 'movie_count']
    + [f'genre_{g.replace(" ","_").replace("-","_")}' for g in TOP_10_GENRES]
    + [f'tag_{t.replace(" ","_").replace("-","_")}' for t in TOP_20_TAGS]
    + [f'cat_{c.replace(" ","_").replace("-","_").replace("/","_")}' for c in KEY_CATEGORIES]
)

# CHECK 1 — No null values
null_counts = df[EXPECTED_COLS].isnull().sum()
total_nulls = null_counts.sum()
if total_nulls == 0:
    print(f'CHECK 1 PASSED: Zero null values in all feature columns ✓')
else:
    print(f'CHECK 1 FAILED: {total_nulls} nulls found')
    print(null_counts[null_counts > 0])

# CHECK 2 — Row count increased
if len(df) > 20383:
    print(f'CHECK 2 PASSED: {len(df):,} rows (more than original 20,383) ✓')
else:
    print(f'CHECK 2 FAILED: Expected more than 20,383 rows, got {len(df):,}')

# CHECK 3 — Label column valid
pos_rate = df[TARGET_COL].mean()
if 0.5 <= pos_rate <= 0.85:
    print(f'CHECK 3 PASSED: Positive class rate = {pos_rate:.3f} ✓')
else:
    print(f'CHECK 3 FAILED: Positive class rate = {pos_rate:.3f} — outside expected range')

# CHECK 4 — All expected columns present
missing_cols = [c for c in EXPECTED_COLS if c not in df.columns]
if not missing_cols:
    print(f'CHECK 4 PASSED: All {len(EXPECTED_COLS)} expected feature columns present ✓')
else:
    print(f'CHECK 4 FAILED: Missing columns: {missing_cols}')

# CHECK 5 — short_description present (needed for Model E)
if COL_DESC in df.columns:
    n_empty_desc = (df[COL_DESC].fillna('').str.strip() == '').sum()
    print(f'CHECK 5 PASSED: short_description present ({n_empty_desc} games have no description) ✓')
else:
    print(f'CHECK 5 FAILED: short_description column missing')

# ── Save ──────────────────────────────────────────────────────────────────────
print()
all_passed = (total_nulls == 0 and len(df) > 20383 and not missing_cols and COL_DESC in df.columns)
if all_passed:
    df.to_csv(OUTPUT_CSV, index=False)
    file_size_mb = OUTPUT_CSV.stat().st_size / (1024 * 1024)
    print(f'Saved: {OUTPUT_CSV}')
    print(f'Size : {file_size_mb:.1f} MB')
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
else:
    print('NOT SAVED — fix the failing checks above first')

CHECK 1 PASSED: Zero null values in all feature columns ✓
CHECK 2 PASSED: 27,750 rows (more than original 20,383) ✓
CHECK 3 PASSED: Positive class rate = 0.685 ✓
CHECK 4 PASSED: All 55 expected feature columns present ✓
CHECK 5 PASSED: short_description present (40 games have no description) ✓

Saved: C:\Users\3214h\Documents\fyp-steam-reception\data\processed\games_features_t4_threshold50.csv
Size : 8.9 MB
Shape: 27,750 rows x 56 columns


## 13. Comparison vs Original (100-review threshold)

In [13]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
print('=' * 72)
print('DATASET COMPARISON')
print('=' * 72)
print()
print(f'  {"Metric":<40} {"NB02 (>=100)":>15} {"NB09 (>=50)":>15} {"Change":>8}')
print('  ' + '-' * 80)

orig_total    = 20383
orig_pos      = 14631
orig_neg      = 5752
orig_pos_rate = orig_pos / orig_total

new_total    = len(df)
new_pos      = df[TARGET_COL].sum()
new_neg      = (df[TARGET_COL] == 0).sum()
new_pos_rate = new_pos / new_total

print(f'  {"Total games":<40} {orig_total:>15,} {new_total:>15,} {new_total-orig_total:>+8,}')
print(f'  {"Well Received (1)":<40} {orig_pos:>15,} {new_pos:>15,} {new_pos-orig_pos:>+8,}')
print(f'  {"Not Well Received (0)":<40} {orig_neg:>15,} {new_neg:>15,} {new_neg-orig_neg:>+8,}')
print(f'  {"Positive rate":<40} {orig_pos_rate*100:>14.1f}% {new_pos_rate*100:>14.1f}% {(new_pos_rate-orig_pos_rate)*100:>+7.1f}%')
print(f'  {"Negative rate":<40} {(1-orig_pos_rate)*100:>14.1f}% {(1-new_pos_rate)*100:>14.1f}% {(orig_pos_rate-new_pos_rate)*100:>+7.1f}%')
print(f'  {"Minority class growth":<40} {"—":>15} {(new_neg-orig_neg)/orig_neg*100:>14.1f}% {"":>8}')
print()

# ── Verdict ───────────────────────────────────────────────────────────────────
print('Verdict:')
if new_pos_rate < orig_pos_rate - 0.01:
    print(f'  Class balance IMPROVED.')
    print(f'  Positive rate dropped from {orig_pos_rate*100:.1f}% to {new_pos_rate*100:.1f}%.')
    print(f'  The lower threshold captured {new_neg - orig_neg:,} additional Not Well Received games.')
    print(f'  Minority class grew by {(new_neg-orig_neg)/orig_neg*100:.1f}%.')
    print()
    print('  Recommendation: Worth running a quick Model D test on this dataset.')
    print('  If Macro F1 improves, consider rerunning NB03-NB07 with MIN_REVIEWS=50.')
elif new_pos_rate > orig_pos_rate + 0.01:
    print(f'  Class balance WORSENED — positive rate increased to {new_pos_rate*100:.1f}%.')
    print(f'  The 50-99 review games are proportionally more Well Received than those already in the dataset.')
    print(f'  Not recommended to proceed with this threshold.')
else:
    print(f'  Class balance is SIMILAR to original ({new_pos_rate*100:.1f}% vs {orig_pos_rate*100:.1f}%).')
    print(f'  The lower threshold added {new_total - orig_total:,} more games but the class ratio is unchanged.')
    print(f'  Still worth testing Model D — more data may help even without better balance.')
print()
print(f'Next step: Load {OUTPUT_CSV.name} in a test notebook,')
print(f'train Model D (depth=8, lr=0.05, iterations=1000, l2=3),')
print(f'and compare Macro F1 against the current baseline of 0.6690.')

DATASET COMPARISON

  Metric                                      NB02 (>=100)     NB09 (>=50)   Change
  --------------------------------------------------------------------------------
  Total games                                       20,383          27,750   +7,367
  Well Received (1)                                 14,631          19,002   +4,371
  Not Well Received (0)                              5,752           8,748   +2,996
  Positive rate                                      71.8%           68.5%    -3.3%
  Negative rate                                      28.2%           31.5%    +3.3%
  Minority class growth                                  —           52.1%         

Verdict:
  Class balance IMPROVED.
  Positive rate dropped from 71.8% to 68.5%.
  The lower threshold captured 2,996 additional Not Well Received games.
  Minority class grew by 52.1%.

  Recommendation: Worth running a quick Model D test on this dataset.
  If Macro F1 improves, consider rerunning NB03-NB07